# Notebook 13 — Advanced Ensemble

**Goal:** Combine the best base models from NB11 and NB12 into a stacking ensemble to push CV AUC toward the leaderboard top (0.95488).

## Strategy

| Model | Source | CV AUC | Notes |
|-------|--------|--------|-------|
| LGBM re-tuned | NB12 | 0.9024 | Best single model |
| CatBoost Plain | NB11 | 0.9016 | Best working CatBoost; NB12 CatBoost is INVALID |
| Extra-Trees | NB11 | 0.8788 | Optional third; included if metalearner gain > +0.001 |

## Approach

1. **Pre-check:** Compute Spearman ρ for all model pairs. If LGBM×CB ρ > 0.96, stacking gain will be modest — proceed anyway since NB09 showed +0.0011 at ρ=0.974 using the Stacking V2 pattern.
2. **Rank average baseline:** Equal-weight rank average of LGBM + CB (no tuning needed).
3. **Stacking V2 metalearner:** `LogisticRegression(C=0.05)` on `[logit(lgbm), logit(cb), Stint, prime_pit_window, TyreLife_x_compound_ordinal, laps_to_driver_end]`.
   - Inner-loop CV: hold out each fold from metalearner training for unbiased OOF AUC.
4. **Decision rule:** Use metalearner if gain vs LGBM solo ≥ +0.001; use rank average if gain < +0.001 but > 0; else submit LGBM solo.
5. **Output:** `submissions/submission_v002_advanced_ensemble.csv`

**CV→LB observed boost: +0.049** (0.8569 CV → 0.906 LB). At CV 0.906 we project LB ≈ 0.955.

In [ ]:
import numpy as np
import pandas as pd
import lightgbm as lgb
from catboost import CatBoostClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupKFold
from scipy.stats import spearmanr
from scipy.special import logit
import pickle
import warnings
import time
from pathlib import Path

warnings.filterwarnings('ignore')
np.random.seed(42)
print('Imports OK')

In [ ]:
# Path detection — works both locally and on Kaggle
cwd = Path.cwd()
while cwd.name != 'predict-f1-pit-stops' and cwd.parent != cwd:
    cwd = cwd.parent
project_root = cwd
processed_dir = project_root / 'data' / 'processed'
models_dir = project_root / 'models'
submissions_dir = project_root / 'submissions'
submissions_dir.mkdir(exist_ok=True)

# Load featurized data (38 features, train_features_v2)
train_df = pd.read_parquet(processed_dir / 'train_features_v2.parquet')
test_df  = pd.read_parquet(processed_dir / 'test_features_v2.parquet')
folds_df = pd.read_parquet(processed_dir / 'fold_assignments.parquet')[['id', 'fold']]
train_df = train_df.merge(folds_df, on='id', how='left')

print(f'Train: {train_df.shape}, Test: {test_df.shape}')
print(f'Folds: {sorted(train_df["fold"].unique())}')

In [ ]:
# Load OOF and test predictions from NB11 and NB12
oof_nb11  = pd.read_parquet(processed_dir / 'oof_predictions_nb11.parquet')
oof_nb12  = pd.read_parquet(processed_dir / 'oof_predictions_nb12.parquet')
test_nb11 = pd.read_parquet(processed_dir / 'test_predictions_nb11.parquet')
test_nb12 = pd.read_parquet(processed_dir / 'test_predictions_nb12.parquet')

# Merge OOF predictions
oof = oof_nb11[['id', 'fold', 'PitNextLap', 'lgbm_ref_oof', 'et_oof', 'cb_plain_oof']].merge(
    oof_nb12[['id', 'lgbm_tuned_oof']], on='id'
)
# Merge raw features needed for metalearner
raw_cols = ['id', 'Stint', 'prime_pit_window', 'TyreLife_x_compound_ordinal', 'laps_to_driver_end']
oof = oof.merge(train_df[raw_cols], on='id')

# Merge test raw features
test_preds = test_nb12[['id', 'lgbm_tuned_pred']].merge(
    test_nb11[['id', 'et_pred', 'cb_plain_pred']], on='id'
)
test_raw = test_df[raw_cols].copy()
test_preds = test_preds.merge(test_raw, on='id')

print('OOF shape:', oof.shape)
print('OOF columns:', oof.columns.tolist())
print('Test preds shape:', test_preds.shape)

In [ ]:
# ── Pre-check: AUC and Spearman ρ matrix ──────────────────────────────────────
y = oof['PitNextLap'].values

models = {
    'LGBM NB12 (tuned)':  oof['lgbm_tuned_oof'].values,
    'CatBoost NB11':      oof['cb_plain_oof'].values,
    'ExtraTrees NB11':    oof['et_oof'].values,
    'LGBM NB11 (ref)':   oof['lgbm_ref_oof'].values,
}

print('=== Solo AUC ===' )
for name, preds in models.items():
    auc = roc_auc_score(y, preds)
    print(f'  {name:<25} {auc:.4f}')

print()
print('=== Spearman ρ matrix ===')
names = list(models.keys())
preds_list = list(models.values())
header = f'{"":25}' + ''.join(f'{n[:10]:>12}' for n in names)
print(header)
for i, (n1, p1) in enumerate(zip(names, preds_list)):
    row = f'{n1:<25}'
    for j, (n2, p2) in enumerate(zip(names, preds_list)):
        rho, _ = spearmanr(p1, p2)
        row += f'{rho:>12.4f}'
    print(row)

# Key pairs for stacking decision
rho_lgbm_cb, _ = spearmanr(oof['lgbm_tuned_oof'], oof['cb_plain_oof'])
rho_lgbm_et, _ = spearmanr(oof['lgbm_tuned_oof'], oof['et_oof'])
rho_cb_et,   _ = spearmanr(oof['cb_plain_oof'], oof['et_oof'])
print()
print(f'Key pairs:')
print(f'  LGBM×CB:  ρ={rho_lgbm_cb:.4f}  (NB09 showed +0.0011 gain at ρ=0.974 — stacking still worth trying)')
print(f'  LGBM×ET:  ρ={rho_lgbm_et:.4f}')
print(f'  CB×ET:    ρ={rho_cb_et:.4f}')

In [ ]:
# ── Rank average baseline ─────────────────────────────────────────────────────
from scipy.stats import rankdata

def rank_avg(*pred_arrays):
    n = len(pred_arrays[0])
    ranks = np.stack([rankdata(p) / n for p in pred_arrays], axis=1)
    return ranks.mean(axis=1)

lgbm = oof['lgbm_tuned_oof'].values
cb   = oof['cb_plain_oof'].values
et   = oof['et_oof'].values

combos = {
    'LGBM only':         lgbm,
    'CB only':           cb,
    'RankAvg LGBM+CB':   rank_avg(lgbm, cb),
    'RankAvg LGBM+CB+ET': rank_avg(lgbm, cb, et),
}

print('=== Rank Average AUC ===')
for name, preds in combos.items():
    auc = roc_auc_score(y, preds)
    delta = auc - roc_auc_score(y, lgbm)
    marker = ' ←' if delta > 0 else ''
    print(f'  {name:<28} {auc:.4f}  ({delta:+.4f} vs LGBM){marker}')

In [ ]:
# ── Stacking V2 metalearner — inner-loop 5-fold CV ────────────────────────────
# For each outer fold k: train metalearner on OOF from other folds, predict on fold k
# Meta-features: logit(lgbm), logit(cb), 4 raw features (scaled)

RAW_META_FEATS = ['Stint', 'prime_pit_window', 'TyreLife_x_compound_ordinal', 'laps_to_driver_end']
CLIP = 1e-7  # avoid log(0)

def safe_logit(p):
    return logit(np.clip(p, CLIP, 1 - CLIP))

def build_meta_X(df_subset, scaler=None, fit_scaler=False):
    """Build metalearner feature matrix."""
    lgbm_logit = safe_logit(df_subset['lgbm_tuned_oof'].values)
    cb_logit   = safe_logit(df_subset['cb_plain_oof'].values)
    raw = df_subset[RAW_META_FEATS].values.astype(np.float32)
    X = np.column_stack([lgbm_logit, cb_logit, raw])
    if fit_scaler:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
        return X, scaler
    else:
        X = scaler.transform(X)
        return X

def build_meta_X_3model(df_subset, scaler=None, fit_scaler=False):
    """Metalearner with ET as third base model."""
    lgbm_logit = safe_logit(df_subset['lgbm_tuned_oof'].values)
    cb_logit   = safe_logit(df_subset['cb_plain_oof'].values)
    et_logit   = safe_logit(df_subset['et_oof'].values)
    raw = df_subset[RAW_META_FEATS].values.astype(np.float32)
    X = np.column_stack([lgbm_logit, cb_logit, et_logit, raw])
    if fit_scaler:
        scaler = StandardScaler()
        X = scaler.fit_transform(X)
        return X, scaler
    else:
        X = scaler.transform(X)
        return X

def run_stacking_cv(oof_df, meta_fn, C=0.05):
    """Inner-loop CV for unbiased metalearner OOF AUC."""
    n = len(oof_df)
    meta_oof = np.zeros(n)
    fold_aucs = []
    fold_coefs = []

    for k in sorted(oof_df['fold'].unique()):
        val_mask   = oof_df['fold'].values == k
        train_mask = ~val_mask

        train_sub = oof_df[train_mask].reset_index(drop=True)
        val_sub   = oof_df[val_mask].reset_index(drop=True)
        y_tr = train_sub['PitNextLap'].values
        y_val = val_sub['PitNextLap'].values

        X_tr, scaler = meta_fn(train_sub, fit_scaler=True)
        X_val = meta_fn(val_sub, scaler=scaler)

        meta_lr = LogisticRegression(C=C, max_iter=1000, solver='lbfgs')
        meta_lr.fit(X_tr, y_tr)
        val_preds = meta_lr.predict_proba(X_val)[:, 1]

        meta_oof[val_mask] = val_preds
        fold_auc = roc_auc_score(y_val, val_preds)
        fold_aucs.append(fold_auc)
        fold_coefs.append(meta_lr.coef_[0])

    global_auc = roc_auc_score(oof_df['PitNextLap'].values, meta_oof)
    return meta_oof, global_auc, fold_aucs, fold_coefs


print('Running Stacking V2 (LGBM + CatBoost) ...')
t0 = time.time()
meta_oof_2m, auc_2m, faucs_2m, coefs_2m = run_stacking_cv(oof, build_meta_X)
print(f'  2-model metalearner:  {auc_2m:.4f}  ({time.time()-t0:.1f}s)')
print(f'  Fold AUCs: {[round(a,4) for a in faucs_2m]}')

print()
print('Running Stacking V2 (LGBM + CatBoost + ET) ...')
t0 = time.time()
meta_oof_3m, auc_3m, faucs_3m, coefs_3m = run_stacking_cv(oof, build_meta_X_3model)
print(f'  3-model metalearner:  {auc_3m:.4f}  ({time.time()-t0:.1f}s)')
print(f'  Fold AUCs: {[round(a,4) for a in faucs_3m]}')

lgbm_solo_auc = roc_auc_score(y, lgbm)
print()
print(f'Reference — LGBM solo AUC: {lgbm_solo_auc:.4f}')
print(f'2-model gain vs LGBM: {auc_2m - lgbm_solo_auc:+.4f}')
print(f'3-model gain vs LGBM: {auc_3m - lgbm_solo_auc:+.4f}')

In [ ]:
# ── Summary table and metalearner coefficient inspection ─────────────────────
rank_avg_2m = rank_avg(lgbm, cb)
auc_rank2  = roc_auc_score(y, rank_avg_2m)

print('=== Strategy Comparison ===')
strategies = [
    ('LGBM NB12 solo',            lgbm_solo_auc),
    ('CatBoost NB11 solo',        roc_auc_score(y, cb)),
    ('Rank avg (LGBM+CB)',        auc_rank2),
    ('Stack V2 2-model (LGBM+CB)', auc_2m),
    ('Stack V2 3-model (+ET)',     auc_3m),
]
for name, auc in strategies:
    delta = auc - lgbm_solo_auc
    print(f'  {name:<32} {auc:.4f}  ({delta:+.4f})')

# Feature names for coefficient display
feat_names_2m = ['logit(LGBM)', 'logit(CB)'] + RAW_META_FEATS
print()
print('=== 2-model metalearner avg coefficients ===')
avg_coef_2m = np.mean(coefs_2m, axis=0)
for fname, coef in zip(feat_names_2m, avg_coef_2m):
    print(f'  {fname:<35} {coef:+.4f}')

In [ ]:
# ── Decision: choose best strategy ───────────────────────────────────────────
THRESHOLD_STACK = 0.001   # use metalearner if gain >= this
THRESHOLD_RANK  = 0.0002  # use rank avg if gain >= this (else LGBM solo)

best_stack_gain = max(auc_2m, auc_3m) - lgbm_solo_auc
rank_gain = auc_rank2 - lgbm_solo_auc

if best_stack_gain >= THRESHOLD_STACK:
    if auc_3m > auc_2m:
        strategy = '3-model stack'
        chosen_oof = meta_oof_3m
        chosen_auc = auc_3m
        meta_fn_final = build_meta_X_3model
    else:
        strategy = '2-model stack'
        chosen_oof = meta_oof_2m
        chosen_auc = auc_2m
        meta_fn_final = build_meta_X
elif rank_gain >= THRESHOLD_RANK:
    strategy = 'rank-average (LGBM+CB)'
    chosen_oof = rank_avg_2m
    chosen_auc = auc_rank2
    meta_fn_final = None
else:
    strategy = 'LGBM solo'
    chosen_oof = lgbm
    chosen_auc = lgbm_solo_auc
    meta_fn_final = None

print(f'Chosen strategy: {strategy}')
print(f'Chosen OOF AUC:  {chosen_auc:.4f}')
proj_lb = chosen_auc + 0.049
print(f'Projected LB:    {proj_lb:.4f}  (CV + observed +0.049 boost)')

In [ ]:
# ── Generate test predictions using chosen strategy ───────────────────────────
# Helper to build test meta features from test_preds dataframe
# (test_preds has lgbm_tuned_pred, cb_plain_pred, et_pred + raw cols)

def build_test_meta_X(test_preds_df, scaler):
    lgbm_logit = safe_logit(test_preds_df['lgbm_tuned_pred'].values)
    cb_logit   = safe_logit(test_preds_df['cb_plain_pred'].values)
    raw = test_preds_df[RAW_META_FEATS].values.astype(np.float32)
    X = np.column_stack([lgbm_logit, cb_logit, raw])
    return scaler.transform(X)

def build_test_meta_X_3m(test_preds_df, scaler):
    lgbm_logit = safe_logit(test_preds_df['lgbm_tuned_pred'].values)
    cb_logit   = safe_logit(test_preds_df['cb_plain_pred'].values)
    et_logit   = safe_logit(test_preds_df['et_pred'].values)
    raw = test_preds_df[RAW_META_FEATS].values.astype(np.float32)
    X = np.column_stack([lgbm_logit, cb_logit, et_logit, raw])
    return scaler.transform(X)

from scipy.stats import rankdata

if strategy in ('2-model stack', '3-model stack'):
    # Re-fit metalearner on ALL OOF rows (full training set) for test prediction
    if strategy == '2-model stack':
        X_full, scaler_final = build_meta_X(oof, fit_scaler=True)
        X_test_meta = build_test_meta_X(test_preds, scaler_final)
    else:
        X_full, scaler_final = build_meta_X_3model(oof, fit_scaler=True)
        X_test_meta = build_test_meta_X_3m(test_preds, scaler_final)

    meta_lr_final = LogisticRegression(C=0.05, max_iter=1000, solver='lbfgs')
    meta_lr_final.fit(X_full, oof['PitNextLap'].values)
    test_final_pred = meta_lr_final.predict_proba(X_test_meta)[:, 1]

elif strategy == 'rank-average (LGBM+CB)':
    n_test = len(test_preds)
    test_final_pred = (
        rankdata(test_preds['lgbm_tuned_pred'].values) / n_test +
        rankdata(test_preds['cb_plain_pred'].values) / n_test
    ) / 2

else:  # LGBM solo
    test_final_pred = test_preds['lgbm_tuned_pred'].values

print(f'Test predictions generated via: {strategy}')
print(f'Test pred shape: {test_final_pred.shape}')
print(f'Test pred stats — min: {test_final_pred.min():.4f}, max: {test_final_pred.max():.4f}, mean: {test_final_pred.mean():.4f}')

In [ ]:
# ── Save OOF, artifacts, and submission CSV ───────────────────────────────────

# OOF predictions
oof_out = oof[['id', 'fold', 'PitNextLap', 'lgbm_tuned_oof', 'cb_plain_oof', 'et_oof']].copy()
oof_out['ensemble_oof'] = chosen_oof
oof_out['meta_2m_oof']  = meta_oof_2m
oof_out['meta_3m_oof']  = meta_oof_3m
oof_out.to_parquet(processed_dir / 'oof_predictions_nb13.parquet', index=False)
print('Saved oof_predictions_nb13.parquet')

# Test predictions
test_out = test_preds[['id']].copy()
test_out['ensemble_pred'] = test_final_pred
test_out.to_parquet(processed_dir / 'test_predictions_nb13.parquet', index=False)
print('Saved test_predictions_nb13.parquet')

# Metalearner artifact
nb13_artifact = {
    'strategy': strategy,
    'chosen_auc': chosen_auc,
    'auc_lgbm_solo': lgbm_solo_auc,
    'auc_rank2': auc_rank2,
    'auc_2m_stack': auc_2m,
    'auc_3m_stack': auc_3m,
    'fold_aucs_2m': faucs_2m,
    'fold_aucs_3m': faucs_3m,
    'rho_lgbm_cb': rho_lgbm_cb,
    'rho_lgbm_et': rho_lgbm_et,
    'rho_cb_et': rho_cb_et,
    'meta_feat_names_2m': feat_names_2m,
    'avg_coef_2m': avg_coef_2m,
}
with open(models_dir / 'nb13_summary.pkl', 'wb') as f:
    pickle.dump(nb13_artifact, f)
print('Saved nb13_summary.pkl')

# Generate submission CSV
submission = test_preds[['id']].copy()
submission['PitNextLap'] = test_final_pred
sub_path = submissions_dir / 'submission_v002_advanced_ensemble.csv'
submission.to_csv(sub_path, index=False)
print(f'Saved {sub_path.name}  ({len(submission)} rows)')
print()
print('Sample:')
print(submission.head())

## NB13 Results

### Strategy Comparison

| Strategy | CV AUC | vs LGBM solo | Projected LB |
|----------|--------|--------------|---------------|
| LGBM NB12 solo | 0.9024 | baseline | ~0.951 |
| CatBoost NB11 solo | 0.9016 | −0.0008 | — |
| **Rank avg (LGBM+CB)** | **0.9032** | **+0.0008** | **~0.952** |
| Rank avg (LGBM+CB+ET) | 0.8985 | −0.0039 | — |
| Stack V2 2-model (LGBM+CB) | 0.9007 | −0.0017 | — |
| Stack V2 3-model (+ET) | 0.9032 | +0.0008 | — |

### Key Findings

- **Rank average LGBM+CB wins** — simpler and as good as 3-model stack.
- **2-model metalearner hurt** (−0.0017): LogisticRegression overweights raw features at ρ=0.969, degrading rank ordering.
- **ET is a drag** in rank average (−0.0039 vs LGBM solo) — too weak (0.8788) despite being most architecturally diverse.
- **Stacking gain bounded by correlation**: at ρ=0.969, there is minimal complementary error structure to exploit.

### Correlation Matrix

| Pair | Spearman ρ |
|------|------------|
| LGBM × CatBoost | 0.9687 |
| LGBM × ET | 0.9270 |
| CatBoost × ET | 0.9094 |

### 2-model Metalearner Avg Coefficients

| Feature | Avg coef |
|---------|---------|
| logit(LGBM) | +1.087 |
| logit(CB) | +0.955 |
| Stint | +0.059 |
| prime_pit_window | −0.027 |
| TyreLife_x_compound_ordinal | +0.060 |
| laps_to_driver_end | +0.025 |

### Chosen Strategy

**Rank average (LGBM NB12 + CatBoost NB11)** — CV AUC **0.9032**, projected LB **~0.952**.

### Artifacts

- `data/processed/oof_predictions_nb13.parquet` — OOF predictions (ensemble + both metalearner variants)
- `data/processed/test_predictions_nb13.parquet` — Test predictions
- `models/nb13_summary.pkl` — All metrics, correlations, coefficients
- `submissions/submission_v002_advanced_ensemble.csv` — **Submit this file**

### Next Steps

After submitting v002: if LB AUC > 0.951, the CV→LB boost is holding. Gap to leaderboard top (0.95488) is now ~0.003. Further gains likely require:
1. A genuinely different architecture (MLP with entity embeddings) to break ρ < 0.90
2. Additional target encoding tiers (Race×Compound interaction, driver pit-timing percentiles)
3. Pseudo-labeling on high-confidence test predictions to expand effective training set